In [1]:
import pandas as pd
import numpy as np

import joblib

from sklearn.pipeline import Pipeline

In [2]:
df = pd.read_csv("../data/employees_final.csv")

In [3]:
import joblib

rf_pipeline = joblib.load("../models/final_model.pkl")

print("Model loaded successfully!")

Model loaded successfully!


In [4]:
type(rf_pipeline)

sklearn.pipeline.Pipeline

In [5]:
employee_ids = df["employee_id"]

df_model = df.drop(
    columns=[
        "employee_id",
        "application_date",
        "last_contact_date"
    ]
)

X = df_model.drop(columns=["enrolled"])

# Remove legacy_propensity_score because this is Model A
X_model_a = X.drop(columns=["legacy_propensity_score"])

predicted_class = rf_pipeline.predict(X_model_a)

predicted_probability = rf_pipeline.predict_proba(X_model_a)[:, 1]

In [6]:
predictions_df = pd.DataFrame({
    "employee_id": employee_ids,
    "predicted_probability": predicted_probability,
    "predicted_enrollment": predicted_class
})

predictions_df.head()

,employee_id,predicted_probability,predicted_enrollment
0,12324,0.00,0
1,17825,0.97,1
2,15200,0.98,1
3,16690,0.92,1
4,17465,0.31,0


In [7]:
import os

print(os.getcwd())
print(os.path.exists("../models/final_model.pkl"))

/Users/varad/Downloads/Froncort/notebooks
True


In [8]:
predictions_df.to_csv("../data/predictions.csv", index=False)

print("predictions.csv saved successfully!")

predictions.csv saved successfully!


In [9]:
predictions_df.shape

(9992, 3)

In [10]:
predictions_df.tail()

,employee_id,predicted_probability,predicted_enrollment
9987,10920,0.19,0
9988,14308,0.90,1
9989,15700,0.99,1
9990,10538,0.99,1
9991,19413,0.98,1


In [11]:
def categorize(prob):
    if prob < 0.30:
        return "High Risk"
    elif prob < 0.60:
        return "Medium Risk"
    else:
        return "Low Risk"

predictions_df["risk_category"] = predictions_df["predicted_probability"].apply(categorize)

predictions_df.head()

,employee_id,predicted_probability,predicted_enrollment,risk_category
0,12324,0.00,0,High Risk
1,17825,0.97,1,Low Risk
2,15200,0.98,1,Low Risk
3,16690,0.92,1,Low Risk
4,17465,0.31,0,Medium Risk


In [12]:
def outreach_action(risk):
    if risk == "High Risk":
        return "Personalized phone call with HR and benefits explanation"
    elif risk == "Medium Risk":
        return "Reminder email with enrollment benefits"
    else:
        return "Standard enrollment notification"

predictions_df["recommended_action"] = predictions_df["risk_category"].apply(outreach_action)

predictions_df.head()

,employee_id,predicted_probability,predicted_enrollment,risk_category,recommended_action
0,12324,0.00,0,High Risk,Personalized phone call with HR and benefits e...
1,17825,0.97,1,Low Risk,Standard enrollment notification
2,15200,0.98,1,Low Risk,Standard enrollment notification
3,16690,0.92,1,Low Risk,Standard enrollment notification
4,17465,0.31,0,Medium Risk,Reminder email with enrollment benefits


In [13]:
predictions_df.to_csv("../data/outreach_assistant.csv", index=False)

In [14]:
region_df = pd.read_csv("../data/region_benefit_profiles.csv")

region_df.head()

,region,n_employees_region,hist_enrollment_rate_region,avg_salary_region,avg_premium_cost_usd,benefits_broker_rating,hr_outreach_capacity,open_enrollment_window_days,state_mandate_level
0,Midwest,2488,0.617,64921.46,595,4.4,469,18,High
1,Northeast,2506,0.612,65008.32,455,3.1,151,38,low
2,South,2424,0.628,65200.49,481,3.3,324,17,MED
3,West,2582,0.613,65007.07,574,4.7,437,28,Low


In [15]:
employee_predictions = df.copy()

employee_predictions["predicted_probability"] = predicted_probability
employee_predictions["predicted_enrollment"] = predicted_class

employee_predictions.head()

,employee_id,age,gender,marital_status,salary,employment_type,region,has_dependents,tenure_years,enrolled,...,n_employees_region,hist_enrollment_rate_region,avg_salary_region,avg_premium_cost_usd,benefits_broker_rating,hr_outreach_capacity,open_enrollment_window_days,state_mandate_level,predicted_probability,predicted_enrollment
0,12324,28,Male,Divorced,44047.60,Full-time,West,No,1.4,0,...,2582,0.613,65007.07,574,4.7,437,28,Low,0.00,0
1,17825,23,Male,Married,80111.28,Full-time,Midwest,Yes,0.5,1,...,2488,0.617,64921.46,595,4.4,469,18,High,0.97,1
2,15200,39,Male,Married,69855.65,Full-time,South,Yes,3.4,1,...,2424,0.628,65200.49,481,3.3,324,17,Medium,0.98,1
3,16690,31,Female,Married,91567.33,Full-time,South,No,1.2,1,...,2424,0.628,65200.49,481,3.3,324,17,Medium,0.92,1
4,17465,42,Female,Single,59861.68,Contract,Midwest,Yes,23.8,0,...,2488,0.617,64921.46,595,4.4,469,18,High,0.31,0


## Tool 1 : Predict Enrollment

In [16]:
def predict_enrollment(employee_id):
    employee = employee_predictions[
        employee_predictions["employee_id"] == employee_id
    ]

    if employee.empty:
        return {
            "Status": "Error",
            "Message": "Employee not found."
        }

    probability = float(employee["predicted_probability"].iloc[0])
    prediction = int(employee["predicted_enrollment"].iloc[0])

    # Confidence level
    if probability >= 0.90 or probability <= 0.10:
        confidence = "High"
    elif probability >= 0.70 or probability <= 0.30:
        confidence = "Medium"
    else:
        confidence = "Low"

    return {
        "Employee ID": int(employee_id),
        "Prediction": "Likely to Enroll" if prediction == 1 else "Not likely to Enroll",
        "Probability": f"{probability:.2%}",
        "Confidence": confidence
    }

In [17]:
employee_predictions["employee_id"].iloc[0]

np.int64(12324)

In [18]:
predict_enrollment(12324)

{'Employee ID': 12324,
 'Prediction': 'Not likely to Enroll',
 'Probability': '0.00%',
 'Confidence': 'High'}

In [19]:
predict_enrollment(17825)

{'Employee ID': 17825,
 'Prediction': 'Likely to Enroll',
 'Probability': '97.00%',
 'Confidence': 'High'}

## Tool 2 : rank_outreach_candidates

In [20]:
def rank_outreach_candidates(region):
    region_info = region_df[
        region_df["region"].str.lower() == region.lower()
    ]

    if region_info.empty:
        return {
            "Status": "Error",
            "Message": "Region not found."
        }

    capacity = int(region_info["hr_outreach_capacity"].iloc[0])

    region_employees = employee_predictions[
        employee_predictions["region"].str.lower() == region.lower()
    ].copy()

    ranked = (
        region_employees
        .sort_values("predicted_probability", ascending=False)
        .head(capacity)
        .copy()
    )

    ranked["Rank"] = range(1, len(ranked) + 1)

    ranked["Probability"] = (
        ranked["predicted_probability"] * 100
    ).round(2).astype(str) + "%"

    ranked["Prediction"] = ranked["predicted_enrollment"].map({
        1: "Likely to Enroll",
        0: "Unlikely to Enroll"
    })

    print(f"\nRegion : {region.title()}")
    print(f"HR Outreach Capacity : {capacity}")
    print(f"Employees Recommended : {len(ranked)}\n")

    return ranked[
        [
            "Rank",
            "employee_id",
            "Probability",
            "Prediction"
        ]
    ]

In [21]:
rank_outreach_candidates("West")


Region : West
HR Outreach Capacity : 437
Employees Recommended : 437



,Rank,employee_id,Probability,Prediction
9985,1,17567,100.0%,Likely to Enroll
1821,2,13565,100.0%,Likely to Enroll
4887,3,12944,100.0%,Likely to Enroll
4882,4,10475,100.0%,Likely to Enroll
4877,5,16564,100.0%,Likely to Enroll
...,...,...,...,...
7890,433,15165,100.0%,Likely to Enroll
7904,434,19657,100.0%,Likely to Enroll
8202,435,18683,100.0%,Likely to Enroll
8158,436,10878,100.0%,Likely to Enroll


In [22]:
rank_outreach_candidates("Midwest")


Region : Midwest
HR Outreach Capacity : 469
Employees Recommended : 469



,Rank,employee_id,Probability,Prediction
1399,1,16435,100.0%,Likely to Enroll
8964,2,11591,100.0%,Likely to Enroll
8883,3,11489,100.0%,Likely to Enroll
7728,4,11046,100.0%,Likely to Enroll
1806,5,12432,100.0%,Likely to Enroll
...,...,...,...,...
4115,465,14311,100.0%,Likely to Enroll
442,466,17691,100.0%,Likely to Enroll
7133,467,19470,100.0%,Likely to Enroll
207,468,14699,100.0%,Likely to Enroll


## Tool 3 : lookup_region_profile()

In [23]:
def lookup_region_profile(region):
    region_info = region_df[
        region_df["region"].str.lower() == region.lower()
    ]

    if region_info.empty:
        return {
            "Status": "Error",
            "Message": "Region not found."
        }

    region_info = region_info.iloc[0]

    return {
        "Region": region_info["region"],
        "Average Salary": f"${region_info['avg_salary_region']:,.2f}",
        "Historical Enrollment Rate": f"{region_info['hist_enrollment_rate_region']:.2%}",
        "Average Premium Cost": f"${region_info['avg_premium_cost_usd']:,.2f}",
        "Benefits Broker Rating": float(round(region_info["benefits_broker_rating"], 2)),
        "HR Outreach Capacity": int(region_info["hr_outreach_capacity"]),
        "Open Enrollment Window (Days)": int(region_info["open_enrollment_window_days"])
    }

In [24]:
lookup_region_profile("West")

{'Region': 'West',
 'Average Salary': '$65,007.07',
 'Historical Enrollment Rate': '61.30%',
 'Average Premium Cost': '$574.00',
 'Benefits Broker Rating': 4.7,
 'HR Outreach Capacity': 437,
 'Open Enrollment Window (Days)': 28}

In [25]:
lookup_region_profile("Midwest")

{'Region': 'Midwest',
 'Average Salary': '$64,921.46',
 'Historical Enrollment Rate': '61.70%',
 'Average Premium Cost': '$595.00',
 'Benefits Broker Rating': 4.4,
 'HR Outreach Capacity': 469,
 'Open Enrollment Window (Days)': 18}

## Tool 4 : explain_prediction()

In [26]:
def explain_prediction(employee_id, requested_features=None):
    """
    Generates a safe, natural-language explanation for an employee's prediction.
    """

    employee = employee_predictions[
        employee_predictions["employee_id"] == employee_id
    ]

    if employee.empty:
        return {
            "Status": "Error",
            "Message": "Employee not found."
        }

    # Refusal rule
    if requested_features is not None:
        forbidden = {"legacy_propensity_score"}

        if any(feature in forbidden for feature in requested_features):
            return {
                "Status": "Refused",
                "Reason": (
                    "The feature 'legacy_propensity_score' was identified as a "
                    "target leakage feature and cannot be used for prediction "
                    "or explanation."
                )
            }

    employee = employee.iloc[0]

    probability = float(employee["predicted_probability"])
    prediction = int(employee["predicted_enrollment"])

    # -----------------------
    # Confidence
    # -----------------------

    if probability >= 0.90 or probability <= 0.10:
        confidence = "High"
    elif probability >= 0.70 or probability <= 0.30:
        confidence = "Medium"
    else:
        confidence = "Low"

    # -----------------------
    # Build explanation
    # -----------------------

    positive_factors = []
    negative_factors = []

    # Salary
    if employee["salary"] >= employee_predictions["salary"].median():
        positive_factors.append("the employee has an above-average salary")
    else:
        negative_factors.append("the employee has a below-average salary")

    # Dependents
    if employee["has_dependents"] == "Yes":
        positive_factors.append("the employee has dependents")
    else:
        negative_factors.append("the employee has no dependents")

    # Tenure
    if employee["tenure_years"] >= 10:
        positive_factors.append("the employee has long tenure with the company")
    elif employee["tenure_years"] <= 2:
        negative_factors.append("the employee is relatively new to the organization")

    # Previous enrollment
    if employee["prior_year_enrolled"] == "Yes":
        positive_factors.append("the employee enrolled in the previous year")
    else:
        negative_factors.append("the employee has no previous enrollment history")

    # Employment type
    if employee["employment_type"] == "Full-Time":
        positive_factors.append("the employee works full-time")

    # Region
    if employee["hist_enrollment_rate_region"] >= 0.70:
        positive_factors.append("the employee belongs to a region with historically high enrollment")
    elif employee["hist_enrollment_rate_region"] <= 0.40:
        negative_factors.append("the employee belongs to a region with relatively low enrollment")

    # Enrollment window
    if employee["open_enrollment_window_days"] >= 20:
        positive_factors.append("there is sufficient time remaining in the enrollment window")

    # Plan tier
    if employee["plan_tier_requested"] != "Unknown":
        positive_factors.append(
            f"the requested plan tier is {employee['plan_tier_requested']}"
        )

    # -----------------------
    # Natural-language explanation
    # -----------------------

    if prediction == 1:

        explanation = (
            f"This employee is predicted to enroll with a probability of "
            f"{probability:.0%}. The prediction is primarily supported because "
            f"{', '.join(positive_factors[:3])}."
        )

        if negative_factors:
            explanation += (
                f" Although {', '.join(negative_factors[:2])}, "
                f"the overall combination of employment and regional factors "
                f"still indicates a high likelihood of enrollment."
            )

    else:

        explanation = (
            f"This employee is predicted not to enroll with a probability of "
            f"{1-probability:.0%}. The prediction is primarily influenced because "
            f"{', '.join(negative_factors[:3])}."
        )

        if positive_factors:
            explanation += (
                f" While {', '.join(positive_factors[:2])}, "
                f"these positive factors were not sufficient to outweigh the "
                f"overall prediction."
            )

    explanation += (
        "\n\nAge, gender, marital status, and "
        "legacy_propensity_score were intentionally excluded from this "
        "explanation to satisfy the project's fairness and leakage requirements."
    )

    return {
        "Employee ID": int(employee_id),
        "Prediction": (
            "Likely to Enroll"
            if prediction == 1
            else "Not Likely to Enroll"
        ),
        "Probability": f"{probability:.2%}",
        "Confidence": confidence,
        "Explanation": explanation
    }

In [27]:
explain_prediction(12324)

{'Employee ID': 12324,
 'Prediction': 'Not Likely to Enroll',
 'Probability': '0.00%',
 'Confidence': 'High',
 'Explanation': "This employee is predicted not to enroll with a probability of 100%. The prediction is primarily influenced because the employee has a below-average salary, the employee has no dependents, the employee is relatively new to the organization. While there is sufficient time remaining in the enrollment window, the requested plan tier is Basic, these positive factors were not sufficient to outweigh the overall prediction.\n\nAge, gender, marital status, and legacy_propensity_score were intentionally excluded from this explanation to satisfy the project's fairness and leakage requirements."}

In [28]:
explain_prediction(
    12324,
    requested_features=["legacy_propensity_score"]
)

{'Status': 'Refused',
 'Reason': "The feature 'legacy_propensity_score' was identified as a target leakage feature and cannot be used for prediction or explanation."}

In [29]:
explain_prediction(1)

{'Status': 'Error', 'Message': 'Employee not found.'}

In [30]:
explain_prediction(17825)

{'Employee ID': 17825,
 'Prediction': 'Likely to Enroll',
 'Probability': '97.00%',
 'Confidence': 'High',
 'Explanation': "This employee is predicted to enroll with a probability of 97%. The prediction is primarily supported because the employee has an above-average salary, the employee has dependents, the requested plan tier is Standard. Although the employee is relatively new to the organization, the employee has no previous enrollment history, the overall combination of employment and regional factors still indicates a high likelihood of enrollment.\n\nAge, gender, marital status, and legacy_propensity_score were intentionally excluded from this explanation to satisfy the project's fairness and leakage requirements."}

In [31]:
lookup_region_profile("Mars")

{'Status': 'Error', 'Message': 'Region not found.'}

In [32]:
employee_predictions[
    (employee_predictions["predicted_probability"] >= 0.70) &
    (employee_predictions["predicted_probability"] < 0.90)
].head(10)

,employee_id,age,gender,marital_status,salary,employment_type,region,has_dependents,tenure_years,enrolled,...,n_employees_region,hist_enrollment_rate_region,avg_salary_region,avg_premium_cost_usd,benefits_broker_rating,hr_outreach_capacity,open_enrollment_window_days,state_mandate_level,predicted_probability,predicted_enrollment
15,11443,63,Female,Married,44205.14,Full-time,West,Yes,1.4,1,...,2582,0.613,65007.07,574,4.7,437,28,Low,0.88,1
36,12171,41,Female,Widowed,65657.44,Full-time,Midwest,No,1.6,1,...,2488,0.617,64921.46,595,4.4,469,18,High,0.86,1
53,18945,53,Male,Married,93148.90,Full-time,West,No,12.3,1,...,2582,0.613,65007.07,574,4.7,437,28,Low,0.85,1
182,17779,61,Female,Married,85051.51,Full-time,West,No,0.1,1,...,2582,0.613,65007.07,574,4.7,437,28,Low,0.81,1
187,16794,55,Female,Single,48995.76,Full-time,Northeast,Yes,2.8,1,...,2506,0.612,65008.32,455,3.1,151,38,Low,0.88,1
193,18756,57,Male,Married,67439.75,Part-time,South,Yes,7.2,1,...,2424,0.628,65200.49,481,3.3,324,17,Medium,0.84,1
229,15710,51,Female,Single,88021.66,Full-time,West,No,2.0,1,...,2582,0.613,65007.07,574,4.7,437,28,Low,0.88,1
247,14495,42,Female,Married,80059.83,Part-time,West,Yes,3.8,1,...,2582,0.613,65007.07,574,4.7,437,28,Low,0.89,1
276,12451,53,Male,Married,33991.40,Full-time,West,Yes,2.1,1,...,2582,0.613,65007.07,574,4.7,437,28,Low,0.89,1
309,14643,28,Female,Single,64357.94,Full-time,Northeast,Yes,4.3,1,...,2506,0.612,65008.32,455,3.1,151,38,Low,0.83,1


In [33]:
explain_prediction(11443)

{'Employee ID': 11443,
 'Prediction': 'Likely to Enroll',
 'Probability': '88.00%',
 'Confidence': 'Medium',
 'Explanation': "This employee is predicted to enroll with a probability of 88%. The prediction is primarily supported because the employee has dependents, there is sufficient time remaining in the enrollment window. Although the employee has a below-average salary, the employee is relatively new to the organization, the overall combination of employment and regional factors still indicates a high likelihood of enrollment.\n\nAge, gender, marital status, and legacy_propensity_score were intentionally excluded from this explanation to satisfy the project's fairness and leakage requirements."}